In [ ]:
"""
Interactive MongoDB Dashboard - Grazioso Salvare Animal Shelter
Course: CS 499 - Computer Science Capstone
Enhancement One: Software Design and Engineering
"""

import os
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Dash and Visualization Components
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
import plotly.express as px

# Import CRUD Module
from CRUD_Python_Module import AnimalShelter

# Configure Jupyter Proxy if running inside Jupyter Environment
JupyterDash.infer_jupyter_proxy_config()

###########################
# Data Manipulation / Model
###########################

# Initialize connection to MongoDB via AnimalShelter CRUD Module
db = AnimalShelter(username="aacuser", password="password123")

# Retrieve initial dataset
raw_data = db.read({})
df = pd.DataFrame.from_records(raw_data)

# Remove MongoDB internal '_id' column to prevent Dash table rendering issues
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Base64 Encoding for Grazioso Salvare Logo Image
image_filename = 'Grazioso Salvare Logo.png'
try:
    with open(image_filename, 'rb') as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode()
    logo_src = f'data:image/png;base64,{encoded_image}'
except FileNotFoundError:
    logo_src = ''  # Fallback if image file is not found

# Rescue Filter Options
filter_options = [
    {'label': 'No Filter (Show All)', 'value': 'all'},
    {'label': 'Water Rescue', 'value': 'water'},
    {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
    {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
]

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display': 'none'}),

    # Header with Logo and Title
    html.Div([
        html.Img(src=logo_src, style={'height': '80px', 'float': 'left', 'margin': '10px'}),
        html.Center(html.B(html.H1('CS-340 Dashboard - Jean Lukenson Collin',
                                   style={'paddingTop': '20px', 'fontFamily': 'Arial, sans-serif'}))),
    ]),

    html.Hr(),

    # Interactive Rescue Type Selection
    html.Div([
        html.H3("Select Rescue Type:", style={'textAlign': 'center', 'fontFamily': 'Arial, sans-serif'}),
        html.Div(
            dcc.RadioItems(
                id='filter-type',
                options=filter_options,
                value='all',
                labelStyle={'display': 'inline-block', 'margin': '10px'},
                style={'textAlign': 'center', 'fontSize': '18px'}
            ),
            style={'textAlign': 'center'}
        ),
    ]),

    html.Hr(),

    # Interactive Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": col, "id": col, "deletable": False, "selectable": True} for col in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        page_current=0,
        page_action='native',
        sort_action='native',
        sort_mode='multi',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto', 'maxHeight': '400px'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '50px',
            'maxWidth': '200px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis',
            'fontFamily': 'Arial, sans-serif'
        },
        style_header={
            'backgroundColor': '#2c3e50',
            'color': 'white',
            'fontWeight': 'bold',
            'textAlign': 'center'
        },
        style_data_conditional=[
            {
                'if': {'row_index': 'odd'},
                'backgroundColor': '#f8f9fa'
            }
        ]
    ),

    html.Br(),
    html.Div(id='selected-row-info', style={'textAlign': 'center', 'fontSize': '16px', 'fontWeight': 'bold', 'margin': '10px'}),
    html.Hr(),

    # Side-by-Side Visualization Components (Pie Chart & Leaflet Map)
    html.Div(className='row',
             style={'display': 'flex', 'flexDirection': 'row'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style={'width': '50%', 'padding': '10px'}
        ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={'width': '50%', 'height': '500px', 'padding': '10px'}
        )
    ])
])

#############################################
# Interaction Callbacks / Controller Logic
#############################################

# Callback 1: Filter Data Table based on Rescue Type selection
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """
    Query MongoDB with appropriate filter criteria based on chosen rescue operation type.
    """
    query = {}

    if filter_type == 'water':
        # Water Rescue Criteria: Dogs, age <= 2 years (104 weeks), specific breeds
        water_breeds = ['Labrador Retriever', 'Newfoundland', 'Golden Retriever']
        query = {
            'animal_type': 'Dog',
            'age_upon_outcome_in_weeks': {'$lte': 104},
            'breed': {'$in': water_breeds}
        }
    elif filter_type == 'mountain':
        # Mountain Rescue Criteria: Dogs, age <= 2 years, specific breeds
        mountain_breeds = ['German Shepherd', 'Belgian Malinois', 'Siberian Husky']
        query = {
            'animal_type': 'Dog',
            'age_upon_outcome_in_weeks': {'$lte': 104},
            'breed': {'$in': mountain_breeds}
        }
    elif filter_type == 'disaster':
        # Disaster Tracking Criteria: Dogs, age <= 2 years, specific breeds
        disaster_breeds = ['German Shepherd', 'Belgian Malinois', 'Golden Retriever']
        query = {
            'animal_type': 'Dog',
            'age_upon_outcome_in_weeks': {'$lte': 104},
            'breed': {'$in': disaster_breeds}
        }

    # Fetch results from database
    results = db.read(query)

    if results:
        filtered_df = pd.DataFrame.from_records(results)
        if '_id' in filtered_df.columns:
            filtered_df.drop(columns=['_id'], inplace=True)
        return filtered_df.to_dict('records')
    else:
        return []

# Callback 2: Dynamic Pie Chart Update based on Filtered Data
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    """
    Generates a Plotly Pie Chart showing distribution of top breeds in current table view.
    """
    if not viewData:
        return html.Div("No data available to display", style={'textAlign': 'center', 'padding': '50px'})

    dff = pd.DataFrame.from_dict(viewData)

    if 'breed' not in dff.columns or dff['breed'].empty:
        return html.Div("No breed data available for plotting", style={'textAlign': 'center', 'padding': '50px'})

    breed_counts = dff['breed'].value_counts().head(10)

    fig = px.pie(
        breed_counts,
        values=breed_counts.values,
        names=breed_counts.index,
        title='Top 10 Breeds in Selected Dataset',
        color_discrete_sequence=px.colors.qualitative.Set3
    )

    fig.update_layout(
        height=400,
        margin={'l': 20, 'r': 20, 't': 40, 'b': 20}
    )

    return [dcc.Graph(figure=fig)]

# Callback 3: Highlight Selected Row
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns:
        return [{
            'if': {'column_id': col},
            'background_color': '#D2F3FF'
        } for col in selected_columns]
    return []

# Callback 4: Selected Row Text Summary Display
@app.callback(
    Output('selected-row-info', 'children'),
    [Input('datatable-id', 'selected_rows'),
     Input('datatable-id', 'data')]
)
def update_selected_row_info(selected_rows, table_data):
    if selected_rows and table_data:
        row_index = selected_rows[0]
        if row_index < len(table_data):
            row_data = table_data[row_index]
            name = row_data.get('name', 'Unknown')
            breed = row_data.get('breed', 'Unknown')
            outcome = row_data.get('outcome_type', 'Unknown')
            return f"Selected Record: {name} | Breed: {breed} | Outcome: {outcome}"
    return "No record selected"

# Callback 5: Geolocation Map Update based on Row Selection
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    """
    Updates the interactive Leaflet Map centering on selected animal's coordinates.
    """
    if not viewData:
        return html.Div("No location data available to display on map",
                        style={'textAlign': 'center', 'padding': '50px'})

    dff = pd.DataFrame.from_dict(viewData)

    row_idx = index[0] if index and index[0] < len(dff) else 0

    # Locate coordinates and descriptive columns
    lat_col = next((col for col in dff.columns if 'lat' in col.lower()), None)
    lon_col = next((col for col in dff.columns if 'lon' in col.lower() or 'long' in col.lower()), None)
    name_col = next((col for col in dff.columns if 'name' in col.lower()), None)
    breed_col = next((col for col in dff.columns if 'breed' in col.lower()), None)

    try:
        lat = float(dff.iloc[row_idx][lat_col]) if lat_col and pd.notna(dff.iloc[row_idx][lat_col]) else 30.75
        lon = float(dff.iloc[row_idx][lon_col]) if lon_col and pd.notna(dff.iloc[row_idx][lon_col]) else -97.48
        animal_name = str(dff.iloc[row_idx][name_col]) if name_col and pd.notna(dff.iloc[row_idx][name_col]) else "Unknown"
        breed = str(dff.iloc[row_idx][breed_col]) if breed_col and pd.notna(dff.iloc[row_idx][breed_col]) else "Unknown"
    except (ValueError, KeyError, IndexError):
        # Default fallbacks centered near Austin, TX
        lat, lon = 30.75, -97.48
        animal_name, breed = "Unknown", "Unknown"

    return [
        dl.Map(style={'width': '100%', 'height': '450px'},
               center=[lat, lon],
               zoom=12,
               children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(position=[lat, lon],
                             children=[
                                 dl.Tooltip(breed),
                                 dl.Popup([
                                     html.H4("Animal Record Details"),
                                     html.P(f"Name: {animal_name}"),
                                     html.Hr(),
                                     html.P(f"Breed: {breed}"),
                                     html.P(f"Coordinates: {lat}, {lon}")
                                 ])
                             ])
               ])
    ]

# Server Execution
if __name__ == '__main__':
    app.run_server(debug=True)

Dash app running on https://slowsimple-floodpresto-3000.codio.io/proxy/8050/
